# 02 — Camera & Feature Tracking

> **참고:** Udacity SFND_2D_Feature_Tracking
> **언어:** C++ (OpenCV), Python (OpenCV)
> **데이터:** KITTI 카메라 이미지 시퀀스

---

## 2-1. 핀홀 카메라 모델



```
[u]   [fx  0  cx] [X/Z]
[v] = [ 0 fy  cy] [Y/Z]
[1]   [ 0  0   1] [ 1 ]

  (u, v)      : 이미지 픽셀 좌표
  (fx, fy)    : 초점 거리 (픽셀 단위)
  (cx, cy)    : 주점 (principal point)
  (X, Y, Z)   : 카메라 좌표계 3D 점
```




**Distortion (왜곡):**
- Radial: `x' = x(1 + k₁r² + k₂r⁴)`, 배럴/핀쿠션 왜곡
- Tangential: `p₁, p₂` 렌즈 기울기



In [ ]:
import cv2, numpy as np

K = np.array([[718.856, 0, 607.193],
              [0, 718.856, 185.216],
              [0,       0,       1]])
dist = np.zeros(5)   # KITTI는 이미 rectified

# 3D → 2D 투영
def project(pts_3d: np.ndarray, K: np.ndarray) -> np.ndarray:
    """pts_3d: (N, 3) 카메라 좌표 → (N, 2) 픽셀"""
    uvw = (K @ pts_3d.T).T
    return uvw[:, :2] / uvw[:, 2:3]




---

## 2-2. 코너 검출

### Harris Corner Detector



```
M = Σ_W [Ix²   IxIy]
        [IxIy   Iy²]

R = det(M) - k·trace(M)²   (k ≈ 0.04)
  R >> 0  : 코너
  R << 0  : 에지
  R ≈ 0   : 평탄
```


In [ ]:
img = cv2.imread("frame.png", cv2.IMREAD_GRAYSCALE)
img_f = np.float32(img)
dst = cv2.cornerHarris(img_f, blockSize=2, ksize=3, k=0.04)
# 임계값 적용
corners = np.where(dst > 0.01 * dst.max())




### FAST (Features from Accelerated Segment Test)

픽셀 p 주변 16개 원형 픽셀에서 연속 N개(N=9)가 p보다 밝거나 어두우면 코너.
- 속도: Harris 대비 10× 빠름
- 실시간 처리에 적합



In [ ]:
fast = cv2.FastFeatureDetector_create(threshold=20, nonmaxSuppression=True)
kps = fast.detect(img)




---

## 2-3. 피처 디스크립터

| 디스크립터 | 방식 | 크기 | 특징 |
|-----------|------|------|------|
| SIFT | Float 128D HOG | 128 × 4B | 회전/스케일 불변, 특허 만료 |
| ORB | Binary 256b | 32B | 빠름, 무료, 회전 불변 |
| BRISK | Binary 512b | 64B | 스케일 불변, GPU 친화적 |
| BRIEF | Binary 256b | 32B | 가장 빠름, 회전 비불변 |

### SIFT 추출



In [ ]:
sift = cv2.SIFT_create(nfeatures=500)
kps, descs = sift.detectAndCompute(img, None)
# kps  : KeyPoint 리스트 (pt, size, angle, response, octave)
# descs: (N, 128) float32




### ORB 추출



In [ ]:
orb = cv2.ORB_create(nfeatures=500)
kps, descs = orb.detectAndCompute(img, None)
# descs: (N, 32) uint8 (binary)




---

## 2-4. 피처 매칭

### Brute-Force (BFMatcher)



In [ ]:
# Float 디스크립터 (SIFT/SURF) → L2 norm
bf = cv2.BFMatcher(cv2.NORM_L2)
matches = bf.knnMatch(descs1, descs2, k=2)

# Lowe's ratio test
good = [m for m, n in matches if m.distance < 0.75 * n.distance]


In [ ]:
# Binary 디스크립터 (ORB/BRISK) → Hamming distance
bf = cv2.BFMatcher(cv2.NORM_HAMMING)
matches = bf.knnMatch(descs1, descs2, k=2)
good = [m for m, n in matches if m.distance < 0.75 * n.distance]




### FLANN (Fast Library for Approximate Nearest Neighbors)



In [ ]:
# SIFT
index_params = dict(algorithm=1, trees=5)  # FLANN_INDEX_KDTREE
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(descs1, descs2, k=2)
good = [m for m, n in matches if m.distance < 0.75 * n.distance]




**BFMatcher vs FLANN:**
- BFMatcher: 정확, 작은 피처셋에 적합 (< 500)
- FLANN: 근사, 빠름, 대용량에 적합 (> 1000)

---

## 2-5. Optical Flow (Lucas-Kanade)



In [ ]:
# 이전 프레임의 코너 → 다음 프레임에서 추적
prev = cv2.imread("frame0.png", cv2.IMREAD_GRAYSCALE)
curr = cv2.imread("frame1.png", cv2.IMREAD_GRAYSCALE)

# Shi-Tomasi 코너 검출
corners = cv2.goodFeaturesToTrack(prev, maxCorners=200, qualityLevel=0.01, minDistance=7)

# LK Optical Flow
lk_params = dict(winSize=(15, 15), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
next_pts, status, err = cv2.calcOpticalFlowPyrLK(prev, curr, corners, None, **lk_params)

# status==1 인 것만 유효한 매치
good_prev = corners[status == 1]
good_curr = next_pts[status == 1]




---

## 2-6. TTC (Time-To-Collision) — Camera 기반

Udacity 3D Object Tracking 프로젝트의 핵심 메트릭.



```
TTC = -dt / (1 - h₀/h₁)

  h₀, h₁ : 이전/현재 프레임에서 물체의 키(픽셀 높이)
  dt      : 프레임 간 시간 간격
```


In [ ]:
def compute_ttc_camera(kps_prev, kps_curr, matches, frame_rate=10.0):
    dt = 1.0 / frame_rate
    dist_ratios = []
    for i, m1 in enumerate(matches):
        for j, m2 in enumerate(matches):
            if i == j:
                continue
            d_prev = np.linalg.norm(kps_prev[m1.queryIdx].pt -
                                    kps_prev[m2.queryIdx].pt)
            d_curr = np.linalg.norm(kps_curr[m1.trainIdx].pt -
                                    kps_curr[m2.trainIdx].pt)
            if d_prev > 100 and abs(d_curr / d_prev - 1.0) < 0.1:
                dist_ratios.append(d_curr / d_prev)
    if not dist_ratios:
        return float('inf')
    median_ratio = np.median(dist_ratios)
    return -dt / (1 - median_ratio)




---

## 2-7. 성능 비교 실험 (Udacity 과제)



In [ ]:
import time

detectors  = ['HARRIS', 'FAST', 'BRISK', 'ORB', 'SIFT']
descriptors = ['BRIEF', 'ORB', 'BRISK', 'FREAK', 'SIFT']

results = []
for det in detectors:
    for desc in descriptors:
        t0 = time.time()
        # ... detect + compute + match ...
        elapsed = time.time() - t0
        results.append({
            'detector': det, 'descriptor': desc,
            'n_kps': len(kps), 'n_matches': len(good),
            'time_ms': elapsed * 1000
        })

import pandas as pd
df = pd.DataFrame(results)
# 추천: FAST + BRIEF (속도), SIFT + FLANN (정확도)




---

## 2-8. C++ 구조 (Udacity 스타일)



```cpp
// matching2D.cpp
void detKeypointsHarris(std::vector<cv::KeyPoint>& keypoints, cv::Mat& img, bool bVis) {
    int blockSize = 2, apertureSize = 3;
    double k = 0.04;
    cv::Mat dst = cv::Mat::zeros(img.size(), CV_32FC1);
    cv::cornerHarris(img, dst, blockSize, apertureSize, k, cv::BORDER_DEFAULT);

    // Non-maximum suppression
    cv::Mat dst_norm;
    cv::normalize(dst, dst_norm, 0, 255, cv::NORM_MINMAX, CV_32FC1);
    double maxOverlap = 0.0;
    for (int r = 0; r < dst_norm.rows; r++) {
        for (int c = 0; c < dst_norm.cols; c++) {
            int response = (int)dst_norm.at<float>(r, c);
            if (response < 100) continue;
            cv::KeyPoint kp(c, r, 2 * apertureSize);
            kp.response = response;
            // ... overlap NMS ...
            keypoints.push_back(kp);
        }
    }
}
```




---

## 핵심 개념 정리

| 개념 | 핵심 |
|------|------|
| Pinhole model | K·[R|t] 투영, 카메라 내부/외부 파라미터 |
| Harris | 고유값 기반 코너 강도, NMS 필요 |
| SIFT | DoG 스케일 공간, 128D HOG 디스크립터 |
| ORB | FAST + BRIEF, binary → Hamming 거리 |
| Ratio test | Lowe's 0.75 규칙, 모호한 매치 제거 |
| LK Flow | 픽셀 intensity 보존 가정, 피라미드로 큰 변위 처리 |

---

## 다음 단계

→ **03_radar_basics.md** : FMCW 신호처리, FFT, CFAR
